### Prompt Chaining Workflow

In [8]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import AzureChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv
import os
import pprint

In [2]:
load_dotenv()
azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")
deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")

model = AzureChatOpenAI(azure_endpoint=azure_endpoint,
    api_key=api_key,
    api_version=api_version,
    deployment_name=deployment_name,)

In [4]:
class BlogState(TypedDict):
    title:str
    outline:str
    content:str


def create_outline(state:BlogState)->BlogState:
    title = state['title']
    prompt = f"Generate an outline for a blog on the topic - {title}"
    outline = model.invoke(prompt)
    state['outline'] = outline 
    return state

def create_blog(state:BlogState)->BlogState:
    title = state['title']
    outline = state['outline']
    prompt = f'Write a detailed blog on the title - {title} using the following outline \n {outline}'
    content = model.invoke(prompt)
    state['content'] = content
    return state

graph = StateGraph(BlogState)

# nodes 
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)

# edges
graph.add_edge(START,"create_outline")
graph.add_edge("create_outline","create_blog")
graph.add_edge("create_blog",END)

workflow = graph.compile()


initial_state = {'title':"Stephen Curry Career"}
final_state = workflow.invoke(initial_state)
print(final_state)

{'title': 'Stephen Curry Career', 'outline': AIMessage(content="### Blog Outline: The Incredible Career of Stephen Curry\n\n#### Introduction\n- Brief overview of Stephen Curry as one of the greatest basketball players in history\n- Importance of his impact on the game and popular culture\n- Purpose of the blog: to explore his career journey, achievements, and influence\n\n#### 1. Early Life and Background\n   - Birth and family background\n     - Born on March 14, 1988, in Akron, Ohio\n     - Influence of his father, Dell Curry, a former NBA player\n   - High school years\n     - Attended Charlotte Christian School\n     - Early signs of talent in basketball\n     - College recruitment challenges\n\n#### 2. College Career at Davidson College\n   - Decision to play for Davidson\n   - Overview of his college stats\n   - Impactful moments and performances\n     - 2008 NCAA Tournament run\n     - Leading Davidson to the Elite Eight\n   - Recognition and awards\n\n#### 3. Entry into the NB

In [10]:
print(final_state["outline"])

content="### Blog Outline: The Incredible Career of Stephen Curry\n\n#### Introduction\n- Brief overview of Stephen Curry as one of the greatest basketball players in history\n- Importance of his impact on the game and popular culture\n- Purpose of the blog: to explore his career journey, achievements, and influence\n\n#### 1. Early Life and Background\n   - Birth and family background\n     - Born on March 14, 1988, in Akron, Ohio\n     - Influence of his father, Dell Curry, a former NBA player\n   - High school years\n     - Attended Charlotte Christian School\n     - Early signs of talent in basketball\n     - College recruitment challenges\n\n#### 2. College Career at Davidson College\n   - Decision to play for Davidson\n   - Overview of his college stats\n   - Impactful moments and performances\n     - 2008 NCAA Tournament run\n     - Leading Davidson to the Elite Eight\n   - Recognition and awards\n\n#### 3. Entry into the NBA\n   - 2009 NBA Draft\n     - Selected 7th overall by 